<a href="https://colab.research.google.com/github/edduDrux/Processamento-de-Imagens/blob/main/calculo_numerico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo Numérico — Métodos para Prova Prática

Este notebook implementa 4 métodos:

1. **Regra do Trapézio** (simples e composta)
2. **Regra de 1/3 de Simpson** (simples e composta)
3. **Eliminação de Gauss** (com multiplicadores)
4. **Gauss-Seidel** (iterativo)

Cada função mostra os **passos intermediários** para você conferir com a planilha. Os exemplos da `prova_prática.xlsx` já estão carregados como teste.

In [ ]:
import numpy as np
import pandas as pd
from math import exp, cos, sin, log, sqrt, pi

# Configurar exibição com mais casas decimais
np.set_printoptions(precision=8, suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:.8f}')

---
## 1. Regra do Trapézio Composta

$$I_1 = \frac{h}{2}\sum_{i=0}^{n} c_i \, y_i \quad\text{com}\quad c_0=c_n=1,\ c_i=2\ (i=1,\dots,n-1),\ h=\frac{b-a}{n}$$

Funciona para **qualquer n ≥ 1**.

In [ ]:
def trapezio(f, a, b, n, mostrar=True):
    """
    Regra do trapézio composta.

    Parâmetros:
        f : função f(x)
        a, b : limites de integração
        n : número de subintervalos (qualquer n >= 1)
        mostrar : se True, exibe a tabela de pontos

    Retorna: valor aproximado da integral
    """
    h = (b - a) / n
    x = [a + i * h for i in range(n + 1)]
    y = [f(xi) for xi in x]

    # Constantes: 1, 2, 2, ..., 2, 1
    c = [1] + [2] * (n - 1) + [1]
    yc = [c[i] * y[i] for i in range(n + 1)]
    soma = sum(yc)
    I = (h / 2) * soma

    if mostrar:
        df = pd.DataFrame({'i': range(n+1), 'x': x, 'y=f(x)': y, 'c_i': c, 'c_i*y_i': yc})
        print(f"h = (b-a)/n = ({b}-{a})/{n} = {h}")
        print(df.to_string(index=False))
        print(f"\nSoma = {soma}")
        print(f"I = (h/2) * soma = ({h}/2) * {soma} = {I}")
    return I

### Teste — Aba `trapézio` da prova prática

$$\int_0^2 \frac{e^{-\cos(x)}}{\sqrt{2x+4}}\, dx \quad\text{com } n=5$$

Resposta esperada da planilha: **I ≈ 0,5644** u.a.

In [ ]:
f1 = lambda x: exp(-cos(x)) / sqrt(2*x + 4)
I = trapezio(f1, a=0, b=2, n=5)

h = (b-a)/n = (2-0)/5 = 0.4
 i          x     y=f(x)  c_i    c_i*y_i
 0 0.00000000 0.18393972    1 0.18393972
 1 0.40000000 0.18170533    2 0.36341067
 2 0.80000000 0.21053781    2 0.42107563
 3 1.20000000 0.27513133    2 0.55026265
 4 1.60000000 0.38372045    2 0.76744090
 5 2.00000000 0.53602529    1 0.53602529

Soma = 2.822154850315442
I = (h/2) * soma = (0.4/2) * 2.822154850315442 = 0.5644309700630884


---
## 2. Regra de 1/3 de Simpson Composta

$$I_2 = \frac{h}{3}\sum_{i=0}^{n} c_i \, y_i \quad\text{com}\quad c_0=c_n=1,\ c_i=4\ \text{se}\ i\ \text{ímpar},\ c_i=2\ \text{se}\ i\ \text{par}$$

**n deve ser múltiplo de 2.**

In [ ]:
def simpson(f, a, b, n, mostrar=True):
    """
    Regra de 1/3 de Simpson composta. n deve ser par.
    """
    if n % 2 != 0:
        raise ValueError(f"n deve ser múltiplo de 2 para Simpson 1/3. Recebido n={n}.")

    h = (b - a) / n
    x = [a + i * h for i in range(n + 1)]
    y = [f(xi) for xi in x]

    # Constantes: 1, 4, 2, 4, 2, ..., 4, 1
    c = [1] + [4 if i % 2 == 1 else 2 for i in range(1, n)] + [1]
    yc = [c[i] * y[i] for i in range(n + 1)]
    soma = sum(yc)
    I = (h / 3) * soma

    if mostrar:
        df = pd.DataFrame({'i': range(n+1), 'x': x, 'y=f(x)': y, 'c_i': c, 'c_i*y_i': yc})
        print(f"h = (b-a)/n = ({b}-{a})/{n} = {h}")
        print(df.to_string(index=False))
        print(f"\nSoma = {soma}")
        print(f"I = (h/3) * soma = ({h}/3) * {soma} = {I}")
    return I

### Teste — Aba `simpson` da prova prática

$$\int_0^3 \frac{xe^{2x}}{(1+2x)^2}\, dx \quad\text{com } n=6$$

Resposta esperada da planilha: **I ≈ 14,1991** u.a.

In [ ]:
f2 = lambda x: (x * exp(2*x)) / (1 + 2*x)**2
I = simpson(f2, a=0, b=3, n=6)

h = (b-a)/n = (3-0)/6 = 0.5
 i          x      y=f(x)  c_i     c_i*y_i
 0 0.00000000  0.00000000    1  0.00000000
 1 0.50000000  0.33978523    4  1.35914091
 2 1.00000000  0.82100623    2  1.64201247
 3 1.50000000  1.88301909    4  7.53207635
 4 2.00000000  4.36785200    2  8.73570401
 5 2.50000000 10.30646938    4 41.22587753
 6 3.00000000 24.69972205    1 24.69972205

Soma = 85.19453331122612
I = (h/3) * soma = (0.5/3) * 85.19453331122612 = 14.199088885204354


### Bônus: integrar uma tabela de pontos (não uma função)

Quando os dados já vêm tabelados (como o exercício do velocímetro).

In [ ]:
def trapezio_tabela(x, y, mostrar=True):
    """Regra do trapézio composta a partir de listas x e y (x igualmente espaçado)."""
    n = len(x) - 1
    h = x[1] - x[0]
    c = [1] + [2] * (n - 1) + [1]
    yc = [c[i] * y[i] for i in range(n + 1)]
    soma = sum(yc)
    I = (h / 2) * soma
    if mostrar:
        df = pd.DataFrame({'i': range(n+1), 'x': x, 'y': y, 'c_i': c, 'c_i*y_i': yc})
        print(df.to_string(index=False))
        print(f"\nh = {h}, soma = {soma}, I = {I}")
    return I

def simpson_tabela(x, y, mostrar=True):
    """Regra de 1/3 de Simpson composta a partir de listas (n deve ser par)."""
    n = len(x) - 1
    if n % 2 != 0:
        raise ValueError("Para Simpson 1/3, n (= len(x)-1) deve ser par.")
    h = x[1] - x[0]
    c = [1] + [4 if i % 2 == 1 else 2 for i in range(1, n)] + [1]
    yc = [c[i] * y[i] for i in range(n + 1)]
    soma = sum(yc)
    I = (h / 3) * soma
    if mostrar:
        df = pd.DataFrame({'i': range(n+1), 'x': x, 'y': y, 'c_i': c, 'c_i*y_i': yc})
        print(df.to_string(index=False))
        print(f"\nh = {h}, soma = {soma}, I = {I}")
    return I

In [ ]:
# Exemplo do velocímetro (slide 32 da Aula 6)
# V em km/h, T em minutos -> distância em km = integral em h
# Conversão: dt em minutos, então dividir por 60 no final
T_min = [0, 5, 10, 15, 20, 25, 30, 35, 40]
V = [23, 25, 30, 35, 40, 45, 47, 52, 60]
T_h = [t/60 for t in T_min]
print("Distância pela regra de Simpson:")
I = simpson_tabela(T_h, V)
print(f"Distância = {I:.4f} km (esperado ~26,25 km)")

Distância pela regra de Simpson:
 i          x  y  c_i  c_i*y_i
 0 0.00000000 23    1       23
 1 0.08333333 25    4      100
 2 0.16666667 30    2       60
 3 0.25000000 35    4      140
 4 0.33333333 40    2       80
 5 0.41666667 45    4      180
 6 0.50000000 47    2       94
 7 0.58333333 52    4      208
 8 0.66666667 60    1       60

h = 0.08333333333333333, soma = 945, I = 26.25
Distância = 26.2500 km (esperado ~26,25 km)


---
## 3. Eliminação de Gauss

Transforma o sistema $[A|b]$ em uma matriz triangular superior usando multiplicadores $m_{ij} = -a_{ij}/a_{ii}$ e em seguida faz a retro-substituição.

In [ ]:
def gauss(A, b, mostrar=True):
    """
    Eliminação de Gauss SEM pivotamento (como na planilha).

    Parâmetros:
        A : matriz dos coeficientes (lista de listas ou np.array)
        b : vetor independente

    Retorna: vetor solução x
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1)
    n = len(b)
    # Matriz aumentada
    M = np.hstack([A, b.reshape(-1, 1)])

    if mostrar:
        print("Matriz aumentada inicial [A|b]:")
        print(M)
        print()

    # Triangularização
    for k in range(n - 1):
        pivo = M[k, k]
        if mostrar:
            print(f"--- Etapa k={k+1}: pivô a[{k+1},{k+1}] = {pivo} ---")
        if abs(pivo) < 1e-14:
            raise ValueError(f"Pivô nulo na linha {k+1}. Use pivotamento ou troque linhas.")
        for i in range(k + 1, n):
            m = -M[i, k] / pivo
            if mostrar:
                print(f"  m{i+1}{k+1} = -a[{i+1},{k+1}]/a[{k+1},{k+1}] = {m}")
            M[i, :] = m * M[k, :] + M[i, :]
        if mostrar:
            print("  Matriz após esta etapa:")
            print(M)
            print()

    # Retro-substituição
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = sum(M[i, j] * x[j] for j in range(i + 1, n))
        x[i] = (M[i, n] - soma) / M[i, i]
        if mostrar:
            print(f"x{i+1} = ({M[i,n]} - {soma}) / {M[i,i]} = {x[i]}")

    if mostrar:
        residuo = b - A @ x
        print(f"\nVetor solução: x = {x}")
        print(f"Resíduo r = b - A·x = {residuo}")
    return x

### Teste — sistema da prova prática (aba `gaus seidel` que na verdade é Eliminação de Gauss)

$$\begin{cases} 15x_1 - 2x_2 + x_3 = 6 \\ x_1 - 15x_2 + x_3 = -4 \\ x_1 + 2x_2 + 25x_3 = 14 \end{cases}$$

Resposta esperada: $x_1 \approx 0{,}4093,\ x_2 \approx 0{,}3284,\ x_3 \approx 0{,}5174$

In [ ]:
A = [[15, -2,  1],
     [ 1, -15, 1],
     [ 1,  2, 25]]
b = [6, -4, 14]

x = gauss(A, b)

Matriz aumentada inicial [A|b]:
[[ 15.  -2.   1.   6.]
 [  1. -15.   1.  -4.]
 [  1.   2.  25.  14.]]

--- Etapa k=1: pivô a[1,1] = 15.0 ---
  m21 = -a[2,1]/a[1,1] = -0.06666666666666667
  m31 = -a[3,1]/a[1,1] = -0.06666666666666667
  Matriz após esta etapa:
[[ 15.          -2.           1.           6.        ]
 [  0.         -14.86666667   0.93333333  -4.4       ]
 [  0.           2.13333333  24.93333333  13.6       ]]

--- Etapa k=2: pivô a[2,2] = -14.866666666666667 ---
  m32 = -a[3,2]/a[2,2] = 0.14349775784753363
  Matriz após esta etapa:
[[ 15.          -2.           1.           6.        ]
 [  0.         -14.86666667   0.93333333  -4.4       ]
 [  0.           0.          25.06726457  12.96860987]]

x3 = (12.968609865470851 - 0) / 25.067264573991032 = 0.5173524150268336
x2 = (-4.4 - 0.4828622540250447) / -14.866666666666667 = 0.32844364937388193
x1 = (6.0 - -0.13953488372093026) / 15.0 = 0.40930232558139534

Vetor solução: x = [0.40930233 0.32844365 0.51735242]
Resíduo r = b - 

---
## 4. Gauss-Seidel (iterativo)

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j<i} a_{ij}\,x_j^{(k+1)} - \sum_{j>i} a_{ij}\,x_j^{(k)}\right)$$

**Critério das linhas** (suficiente para garantir convergência): $|a_{ii}| \ge \sum_{j\ne i} |a_{ij}|$ para cada linha $i$.

**Critério de parada**: $\max_i |x_i^{(k+1)} - x_i^{(k)}| < \text{tol}$

In [ ]:
def criterio_linhas(A, mostrar=True):
    """Verifica se a matriz A satisfaz o critério das linhas."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    ok = True
    if mostrar:
        print("Critério das linhas (|a_ii| >= soma dos |a_ij|, j != i):")
    for i in range(n):
        soma = sum(abs(A[i, j]) for j in range(n) if j != i)
        passa = abs(A[i, i]) >= soma
        if mostrar:
            simbolo = ">=" if passa else "<"
            print(f"  Linha {i+1}: |{A[i,i]}| = {abs(A[i,i])} {simbolo} {soma}  -> {'OK' if passa else 'FALHA'}")
        if not passa:
            ok = False
    if mostrar:
        print("  -> Convergência GARANTIDA." if ok else "  -> Critério NÃO satisfeito (pode ou não convergir).")
    return ok


def gauss_seidel(A, b, x0, tol=1e-2, kmax=50, mostrar=True):
    """
    Método iterativo de Gauss-Seidel.

    Parâmetros:
        A : matriz dos coeficientes
        b : vetor independente
        x0 : aproximação inicial
        tol : tolerância (critério: max |x^(k+1) - x^(k)| < tol)
        kmax : número máximo de iterações

    Retorna: (x, k, tabela_de_iteracoes)
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1)
    x = np.array(x0, dtype=float).reshape(-1)
    n = len(b)

    if mostrar:
        criterio_linhas(A)
        print()

    # Tabela: cada coluna é uma iteração k
    historico = {0: x.copy()}
    diferencas = {0: None}

    for k in range(1, kmax + 1):
        x_novo = x.copy()
        for i in range(n):
            soma = sum(A[i, j] * x_novo[j] for j in range(i)) \
                 + sum(A[i, j] * x[j] for j in range(i + 1, n))
            x_novo[i] = (b[i] - soma) / A[i, i]

        max_dif = max(abs(x_novo[i] - x[i]) for i in range(n))
        historico[k] = x_novo.copy()
        diferencas[k] = max_dif
        x = x_novo

        if max_dif < tol:
            break

    # Monta tabela como na planilha
    tabela = pd.DataFrame(historico, index=[f'x{i+1}' for i in range(n)])
    tabela.loc['max dif'] = [diferencas[col] if diferencas[col] is not None else np.nan for col in tabela.columns]
    tabela.loc['CP'] = ['-' if diferencas[col] is None else ('para' if diferencas[col] < tol else 'continua') for col in tabela.columns]

    if mostrar:
        print(f"Tabela de iterações (tol = {tol}):")
        print(tabela.to_string())
        print(f"\nConvergiu em k = {k} iterações.")
        print(f"Solução: x = {x}")
        residuo = b - A @ x
        print(f"Resíduo r = b - A·x = {residuo}")
    return x, k, tabela

### Teste — mesmo sistema da prova prática (aba `gaus`)

Com $x^{(0)} = [1, 0, 1]^T$ e tol $= 0{,}01$.

Esperado: convergir em **k = 3** iterações com solução $\approx [0{,}4094;\ 0{,}3284;\ 0{,}5174]$

In [ ]:
A = [[15, -2,  1],
     [ 1, -15, 1],
     [ 1,  2, 25]]
b = [6, -4, 14]
x0 = [1, 0, 1]

x, k, tabela = gauss_seidel(A, b, x0, tol=0.01, kmax=20)

Critério das linhas (|a_ii| >= soma dos |a_ij|, j != i):
  Linha 1: |15.0| = 15.0 >= 3.0  -> OK
  Linha 2: |-15.0| = 15.0 >= 2.0  -> OK
  Linha 3: |25.0| = 25.0 >= 3.0  -> OK
  -> Convergência GARANTIDA.

Tabela de iterações (tol = 0.01):
                 0          1          2          3
x1      1.00000000 0.33333333 0.41285926 0.40935273
x2      0.00000000 0.35555556 0.32873877 0.32843595
x3      1.00000000 0.51822222 0.51718653 0.51735101
max dif        NaN 0.66666667 0.07952593 0.00350653
CP               -   continua   continua       para

Convergiu em k = 3 iterações.
Solução: x = [0.40935273 0.32843595 0.51735101]
Resíduo r = b - A·x = [-0.00077012 -0.00016449  0.        ]


### Teste — exemplo da Aula 5 (slide 12)

$$\begin{cases} 10x_1 + 2x_2 + 3x_3 = 7 \\ x_1 + 5x_2 + x_3 = -8 \\ 2x_1 + 3x_2 + 10x_3 = 6 \end{cases}$$

Esperado: $x \approx [0{,}7826;\ -1{,}9631;\ 1{,}0324]$

In [ ]:
A = [[10, 2,  3],
     [ 1, 5,  1],
     [ 2, 3, 10]]
b = [7, -8, 6]
x0 = [0, 0, 0]

x, k, tabela = gauss_seidel(A, b, x0, tol=0.01, kmax=20)

Critério das linhas (|a_ii| >= soma dos |a_ij|, j != i):
  Linha 1: |10.0| = 10.0 >= 5.0  -> OK
  Linha 2: |5.0| = 5.0 >= 2.0  -> OK
  Linha 3: |10.0| = 10.0 >= 5.0  -> OK
  -> Convergência GARANTIDA.

Tabela de iterações (tol = 0.01):
                 0           1           2           3           4
x1      0.00000000  0.70000000  0.75340000  0.77938280  0.78264516
x2      0.00000000 -1.74000000 -1.94708000 -1.96256536 -1.96310764
x3      0.00000000  0.98200000  1.03344400  1.03289305  1.03240326
max dif        NaN  1.74000000  0.20708000  0.02598280  0.00326236
CP               -    continua    continua    continua        para

Convergiu em k = 4 iterações.
Solução: x = [ 0.78264516 -1.96310764  1.03240326]
Resíduo r = b - A·x = [0.00255392 0.00048979 0.        ]


---
## Resumo rápido — como usar na prova

**Integração de função:**
```python
f = lambda x: ...  # sua função aqui
I = trapezio(f, a=0, b=2, n=5)
I = simpson(f, a=0, b=3, n=6)
```

**Integração de tabela de pontos:**
```python
x = [...]  # pontos
y = [...]  # valores
I = trapezio_tabela(x, y)
I = simpson_tabela(x, y)  # n par
```

**Sistema linear:**
```python
A = [[...], [...], [...]]
b = [...]
x = gauss(A, b)                                        # direto
x, k, tab = gauss_seidel(A, b, x0=[0,0,0], tol=0.01)   # iterativo
```

**Funções matemáticas disponíveis** (já importadas do `math`):
`exp, log, sqrt, sin, cos, tan, pi`

**Atenção a casos comuns:**
- Simpson 1/3 exige `n` par (múltiplo de 2)
- Gauss-Seidel: se o critério das linhas falhar, tente **reordenar as linhas** para colocar os maiores elementos na diagonal
- Se uma função tem $\log(x)$ ou $\sqrt{x}$, cuidado com $x=0$ no limite inferior